# 03 · Benchmark design: audit and answer key

1. **Audit** the LLM labels against human labels (target ≥ 90% agreement).
2. **Generate** the questions and compute exact answers from the labels with pandas.

The answer key is only as good as the labels, which is why the audit comes first.

In [1]:
import json
from pathlib import Path

import pandas as pd
from rlm_logistics.questions import build_questions

ROOT = Path("..")
labels = pd.read_csv(ROOT / "data/labels/labels.csv")
meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
records = pd.read_csv(ROOT / "data/processed/records.csv").merge(labels, on="order_id")
print(f"{len(records):,} labeled records")

17,030 labeled records


## 1. Human audit
`data/labels/audit_sheet.csv` has 200 random reviews with an LLM translation, but **not** the LLM's label (a *blind* audit),
so the human isn't nudged toward agreeing. Fill `human_label` with one of: delivery, product, seller_service, positive, other.

In [2]:
audit = pd.read_csv(ROOT / "data/labels/audit_sheet.csv", encoding="utf-8-sig").merge(labels, on="order_id")
audit["human_label"] = audit.human_label.str.strip().str.lower()
done = audit[audit.human_label.notna()]
if done.empty:
    print("Audit not filled in yet: fill human_label in data/labels/audit_sheet.csv, then re-run.")
else:
    agreement = (done.human_label == done.label).mean()
    print(f"agreement: {agreement:.1%} on {len(done)} reviews -> {'PASS' if agreement >= 0.9 else 'FIX THE PROMPT'}")
    if "human_label_original" in done:  # rows re-labeled with the 6 agreed rules (see edited_by_rule)
        print(f"before rule edits: {(done.human_label_original.str.strip().str.lower() == done.label).mean():.1%}, "
              f"{done.edited_by_rule.sum()} human labels edited by rule")
    display(pd.crosstab(done.human_label, done.label, rownames=["human"], colnames=["LLM"]))

agreement: 93.0% on 199 reviews -> PASS
before rule edits: 83.4%, 19 human labels edited by rule


LLM,delivery,other,positive,product,seller_service
human,,,,,
delivery,22,1,0,0,0
other,0,6,1,0,0
positive,0,0,126,1,0
product,1,1,2,18,2
seller_service,4,1,0,0,13


Disagreements are the most useful rows to read: they show whether the label definitions are unclear or the model is wrong.

In [3]:
if not done.empty:
    display(done[done.human_label != done.label][["english", "human_label", "label"]])

,english,human_label,label
17,"Very small, ideal for carrying in the car.",product,positive
24,"Sorry, I liked the product, but they should of...",product,other
26,Product with missing items.,product,seller_service
41,The delivery came missing merchandise. I only ...,seller_service,delivery
50,I made a mistake by not paying attention that ...,positive,product
65,Ok,other,positive
72,I am going to comment on a lack of professiona...,seller_service,other
78,I cannot yet evaluate the product since I have...,product,positive
88,The product took far too long to be delivered....,product,delivery
124,They send the product to be picked up at the p...,delivery,other


## 2. Questions and answer key
* **T1 lookup**: the complaint of one order. Work stays constant as the context grows.
* **T2 aggregation**: a percentage over all matching orders. Work grows linearly.
* **T3 pairwise**: every pair of orders meeting a condition. Work grows quadratically.
  Its scope (all orders or one state) is chosen so each answer has 3-60 pairs, which keeps answers checkable.

6 questions per tier at 32K/128K/512K, plus 2 per tier at 2M.

In [4]:
contexts = {name: records.iloc[:m["n_records"]] for name, m in meta.items()}
qs = build_questions(contexts, {"32k": 6, "128k": 6, "512k": 6, "2m": 2})
qs.to_json(ROOT / "data/processed/questions.jsonl", orient="records", lines=True, force_ascii=False)
qs.groupby(["size", "tier"]).size().unstack()

tier,T1,T2,T3
size,,,
128k,6,6,6
2m,2,2,2
32k,6,6,5
512k,6,6,6


The label mix should be similar at every size (same random order), so accuracy differences come from size, not content.

In [5]:
pd.DataFrame({name: ctx.label.value_counts(normalize=True).round(3) for name, ctx in contexts.items()})

,32k,128k,512k,2m
label,,,,
delivery,0.126,0.133,0.132,0.120
other,0.045,0.042,0.046,0.048
positive,0.613,0.616,0.618,0.624
product,0.115,0.125,0.117,0.123
seller_service,0.100,0.084,0.088,0.085


In [6]:
for r in qs.groupby("tier").head(1).itertuples():
    print(f"[{r.id}] {r.question}\n  -> {str(r.answer)[:120]}\n")

[32k-T1-0] What is the main complaint in the review of order O39999? Answer with exactly one of: delivery, product, seller_service, positive, other.
  -> delivery

[32k-T2-0] Among orders delivered after the estimated date, what percentage of reviews count as a delivery complaint (late, lost, not received, damaged in transit)? Answer with a number rounded to one decimal.
  -> 72.4

[32k-T3-0] Among all orders, list every pair of orders from the same customer city where one review is a delivery complaint (late, lost, not received, damaged in transit) and the other is a product complaint (quality, defect, not as described). Answer as a list of order-id pairs, e.g. [[id1, id2], [id3, id4]].
  -> [['O06174', 'O13979'], ['O06174', 'O67567'], ['O06174', 'O74372'], ['O06174', 'O87234'], ['O07051', 'O37574'], ['O07051'



## 3. Sanity check: can pandas alone answer these?
Only if it has the labels. Without them, every T2 and T3 answer needs a judgment about review text,
which is exactly the work RAG and RLM are tested on (PRD risk: *"questions answerable by pandas alone"*).

In [7]:
print(qs[qs.tier == "T3"].answer.map(len).describe()[["min", "50%", "max"]])

min     3.0
50%    11.0
max    58.0
Name: answer, dtype: float64
